# 01 — Data Cleaning and Feature Engineering

This notebook loads the Occuspace Rec Center export, applies cleaning rules, engineers time and academic-calendar features, assigns a time-based train/validation/test split, and saves `data/rec_center_clean.parquet` for downstream modeling.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if (ROOT / "rec_center_utils.py").exists():
    pass
elif (ROOT / "rec_center" / "rec_center_utils.py").exists():
    ROOT = ROOT / "rec_center"
elif (ROOT.parent / "rec_center_utils.py").exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("Could not locate rec_center_utils.py")
sys.path.insert(0, str(ROOT))

from rec_center_utils import (
    TRAIN_END,
    VAL_END,
    clean_data,
    data_path,
    figures_path,
    load_clean_data,
    load_raw_data,
    save_clean_data,
)

sns.set_theme(style="whitegrid", context="notebook")


## Load raw data

Source workbook: `datasets/rec_center_usage.xlsx` (~223k rows, six locations, 30-minute intervals).


In [ ]:

raw = load_raw_data()
raw.head()



## Cleaning decisions

- Drop duplicate `location` + `timestamp` pairs (keep first after sorting).
- Cap `average_utilization` at 1.0 for modeling while retaining the raw value.
- Exclude peak occupancy/utilization from predictors in modeling notebooks to avoid same-interval leakage.
- Engineer cyclical time features and Cal Poly academic calendar flags.


In [ ]:

cleaned, report = clean_data(raw)
report



In [ ]:

cleaned[["location", "timestamp", "average_utilization", "average_utilization_raw", "usage_level", "split"]].head()



## Time-based split

- **Train:** through 2024-06-30
- **Validation:** through 2024-12-31
- **Test:** remaining recent months (held out for final evaluation)


In [ ]:

cleaned["split"].value_counts()



In [ ]:

output_path = save_clean_data(cleaned)
output_path

